# 🚀 Advanced Fintech Technical Interview Cases (Senior / Staff Level)

Welcome to the **Advanced Fintech & Data Science Interview Notebook**.

These scenarios test advanced capabilities demanded by tier-1 fintech companies (**American Express, Stripe, Visa, Block, Goldman Sachs, Brex**). They require handling **diverse enterprise file formats** (`.json`, `.jsonl`, `.xml`, `.tsv`, `.psv`, `.dat`, `.yaml`, `.csv`), complex multi-table relational logic, window functions, and defensive data cleaning.

---
### 📁 Available Enterprise Datasets in `./data/`
- `data/customers.csv` (Customer demographics, KYC status, credit score, income, balances, account tier, PEP flag)
- `data/raw_transactions.csv` (Amounts, card types, statuses, device types, fraud flags, dates, regions)
- `data/merchants.csv` (Merchant categories, fee structures, risk levels, monthly volume)
- `data/disputes.csv` (Dispute records, chargebacks, fee tracking, resolution dates)
- `data/api_event_logs.json` (Real-time REST API Security & MFA Webhooks)
- `data/device_telemetry.jsonl` (Streaming Mobile App Sessions & Telemetry)
- `data/fx_rates_daily.tsv` (Daily Multi-Currency Forex Market Rates)
- `data/kyc_audit_records.psv` (Pipe-Delimited Identity Verification Logs)
- `data/credit_bureau_scores.xml` (Experian/Equifax FICO & Credit Bureau Pulls)
- `data/ach_clearing_settlement.dat` (NACHA Electronic ACH Batch Clearing Files)
- `data/aml_sanctions_watchlist.yaml` (OFAC Sanctions & High-Risk IP Prefixes)

---
### 🧠 Apply the Universal Framework:
```
1. Define Target Output (Columns, 1-Row Grain, Visual Shape)
2. Raw Data Audit (Inspect Dtypes, Formats, Nulls, Join Cardinality)
3. Backward Transformation Pipeline (Filter ➔ Cast ➔ Merge ➔ Group ➔ Derive)
4. Sanity Verification (Assert Uniqueness, Valid Bounds, Null Checks)
```

In [2]:
# Setup environment & verify multi-format files
import pandas as pd
import numpy as np
import json
import yaml
import os

DATA_DIR = 'data'
print('✅ Available Enterprise Data Files:', os.listdir(DATA_DIR))

✅ Available Enterprise Data Files: ['ach_clearing_settlement.dat', 'aml_sanctions_watchlist.yaml', 'api_event_logs.json', 'credit_bureau_scores.xml', 'customers.csv', 'device_telemetry.jsonl', 'disputes.csv', 'fx_rates_daily.tsv', 'kyc_audit_records.psv', 'merchants.csv', 'raw_transactions.csv']


---
## 🏢 Case 1 (TSV Forex Integration & Multi-Currency Normalization): Cross-Border FX Revenue Margin

### 📌 Business Scenario
> **Stakeholder:** Head of Global Treasury & Foreign Exchange  
> *"Our international transactions are billed across multiple currency corridors. Load `raw_transactions.csv` and our daily Forex feed `fx_rates_daily.tsv` (Tab-Separated). Convert completed foreign transactions to base USD using the matching date's spot rate. Calculate our Forex spread margin revenue by comparing the bid/ask spread against the central bank fixing rate, and identify the top 3 highest FX revenue generating regions."*

In [15]:
# ✍️ YOUR CODE HERE FOR CASE 1:
import pandas as pd
import numpy as np

tx=pd.read_csv('data/raw_transactions.csv')
fx=pd.read_csv('data/fx_rates_daily.tsv', sep='\t')

print("Transaction shape:", tx.shape)
print("FX Rates columns:", fx.columns.tolist())
print("Sample TSV row:\n", fx.head(1))

# Step 3.1: Pipeline - Ingest & type-cast dates & strings
tx['transaction_date']=pd.to_datetime(tx['transaction_date'], format='mixed', errors='coerce').dt.normalize()
tx['transaction_status']=tx['transaction_status'].astype(str).str.strip().str.title()
tx['region']=tx['region'].astype(str).str.strip().str.title()
tx['transaction_amount']=pd.to_numeric(tx['transaction_amount'], errors='coerce').fillna(0)

fx['rate_date']=pd.to_datetime(fx['rate_date'], errors='coerce').dt.normalize()
fx['spot_rate']=pd.to_numeric(fx['spot_rate'], errors='coerce')
fx['bid_rate']=pd.to_numeric(fx['bid_rate'], errors='coerce')
fx['ask_rate']=pd.to_numeric(fx['ask_rate'], errors='coerce')

# Step 3.2: Pipeline - Calculate daily average FX spread margin ratio
fx['spread_ratio'] = (fx['ask_rate'] - fx['bid_rate']) / fx['spot_rate']
daily_fx = fx.groupby('rate_date', as_index=False).agg(
    daily_spread_margin=('spread_ratio', 'mean')
)

# Step 3.3: Pipeline - Filter Completed transactions & merge with daily FX rates
completed_tx=tx[tx['transaction_status'] == 'Completed'].copy()
merged_tx=completed_tx.merge(daily_fx, left_on='transaction_date', right_on='rate_date', how='left')

# Handle dates without FX records using overall median spread
median_spread=daily_fx['daily_spread_margin'].median()
merged_tx['daily_spread_margin']=merged_tx['daily_spread_margin'].fillna(median_spread)

# Step 3.4: Pipeline - Derive FX revenue per transaction
merged_tx['fx_revenue_usd'] = merged_tx['transaction_amount'] * merged_tx['daily_spread_margin']

# Step 3.5: Pipeline - Group by region & aggregate
top3_fx_regions=(
    merged_tx
    .groupby('region', as_index=False)
    .agg(
        total_volume_usd=('transaction_amount', 'sum'),
        total_fx_revenue_usd=('fx_revenue_usd', 'sum')
    )
    .assign(
        fx_margin_pct=lambda d: ((d['total_fx_revenue_usd']/d['total_volume_usd']) * 100).round(4),
        total_volume_usd=lambda d: d['total_volume_usd'].round(2),
        total_fx_revenue_usd=lambda d: d['total_fx_revenue_usd'].round(2)
    )
    .sort_values(by='total_fx_revenue_usd', ascending=False)
    .head(3)
    .reset_index(drop=True)   
)

# Step 4.1: Sanity audit - Grain & length checks
assert top3_fx_regions['region'].is_unique, "Duplicate regions found!"
assert len(top3_fx_regions) == 3, "Output row count is not 3!"

# Step 4.2: Sanity audit - Financial boundary checks
assert (top3_fx_regions['total_fx_revenue_usd'] > 0).all(), "Negative or zero FX revenue found!"
assert ((top3_fx_regions['fx_margin_pct'] > 0) & (top3_fx_regions['fx_margin_pct'] < 5.0)).all(), "Unrealistic FX margin %!"

# Step 4.3: Sanity audit - Missing value checks
assert top3_fx_regions.isna().sum().sum() == 0, "Unexpected nulls in output!"
print("✅ Case 1 complete and verified!")
display(top3_fx_regions)

Transaction shape: (15000, 11)
FX Rates columns: ['rate_date', 'base_currency', 'quote_currency', 'spot_rate', 'bid_rate', 'ask_rate', 'central_bank_fixing']
Sample TSV row:
     rate_date base_currency quote_currency  spot_rate  bid_rate  ask_rate  \
0  2025-01-01           USD            EUR     0.9007    0.9003    0.9011   

   central_bank_fixing  
0               0.9007  
✅ Case 1 complete and verified!


,region,total_volume_usd,total_fx_revenue_usd,fx_margin_pct
0,East,928171.99,1155.26,0.1245
1,South,913812.89,1137.33,0.1245
2,North,891326.33,1108.81,0.1244


---
## 🏢 Case 2 (Nested JSON Flattening & ATO Detection): Impossible Travel & High-Risk Security Webhooks

### 📌 Business Scenario
> **Stakeholder:** VP of Information Security & Fraud Intelligence  
> *"Fraudsters are attempting Account Takeovers (ATO) via automated bots and credential stuffing. Ingest our nested webhook logs `api_event_logs.json` using `pd.json_normalize()`. Flag all security events where `security_flags.risk_score >= 0.85` OR (`security_flags.mfa_prompted == True` AND `security_flags.mfa_passed == False`). Join with `customers.csv` to identify customer accounts under active attack and aggregate total account balance exposure by customer `account_tier`."*

In [ ]:
# ✍️ YOUR CODE HERE FOR CASE 2:
import json
import pandas as pd
import numpy as np

# Step 2.1: Raw data audit - Source ingestion & JSON flattening
with open('data/api_event_logs.json') as f:
    api_payload=json.load(f)

events_df=pd.json_normalize(api_payload['data'])
cust_df=pd.read_csv('data/customers.csv')

print("EVents flattened shape:",events_df.shape)
print("Flattened columns sample:",[c for c in events_df.columns if 'security_flags' in c])
print("Customers shape:",cust_df.shape)

# Step 3.1: Pipeline - Ingest & type-cast
events_df['customer_id']=events_df['customer_id'].astype(str).str.strip()
events_df['security_flags.risk_score']=pd.to_numeric(events_df['security_flags.risk_score'],errors='coerce').fillna(0)
events_df['security_flags.mfa_prompted']=events_df['security_flags.mfa_prompted'].astype(bool)
events_df['security_flags.mfa_passed']=events_df['security_flags.mfa_passed'].astype(bool)

cust_df['customer_id']=cust_df['customer_id'].astype(str).str.strip()
cust_df['account_tier']=cust_df['account_tier'].astype(str).str.strip().str.title()
cust_df['account_balance']=pd.to_numeric(cust_df['account_balance'],errors='coerce').fillna(0)

# Step 3.2: Pipeline - Filter high-risk security attack events
is_high_risk=events_df['security_flags.risk_score'] >= 0.85
is_mfa_failed=(events_df['security_flags.mfa_prompted'] == True) & (events_df['security_flags.mfa_passed'] == False)
flagged_events=events_df[is_high_risk | is_mfa_failed].copy()

# Step 3.3: Pipeline - Pre-aggregate to unique customer grain (prevents balance duplication)
attacked_customers=flagged_events.groupby('customer_id', as_index=False).agg(
        attack_event_count=('event_id','count'),
        mean_attack_risk=('security_flags.risk_score', 'mean')
)

# Step 3.4: Pipeline - Merge with customer demographics & balances
merged_risk_df=attacked_customers.merge(
        cust_df[['customer_id','account_tier','account_balance']],
        on='customer_id',
        how='inner'
    )

# Step 3.5: Pipeline - Group by account_tier & derive tier exposure
tier_risk_summary=(merged_risk_df
    .groupby('account_tier', as_index=False)
    .agg(
        attacked_customer_count=('customer_id','count'),
        total_balance_at_risk_usd=('account_balance','sum'),
        avg_attack_risk_score=('mean_attack_risk','mean')
    )
    .assign(
        total_balance_at_risk_usd=lambda d: d['total_balance_at_risk_usd'].round(2),
        avg_attack_risk_score=lambda d: d['avg_attack_risk_score'].round(2)
    )
    .sort_values(by='total_balance_at_risk_usd', ascending=False)
    .reset_index(drop=True)
    )

# Step 4.1: Sanity audit - Grain uniqueness check
assert tier_risk_summary['account_tier'].is_unique, "Duplicate account tiers found!"

# Step 4.2: Sanity audit - Range & boundary invariants
assert (tier_risk_summary['total_balance_at_risk_usd'] >= 0).all(), "Negative balance found!"
assert tier_risk_summary['avg_attack_risk_score'].between(0.0, 1.0).all(), "Risk score out of [0, 1] bounds!"

# Step 4.3: Sanity audit - Conservation of unique attacked customers
assert tier_risk_summary['attacked_customer_count'].sum() == len(merged_risk_df), "Customer count mismatch!"
assert tier_risk_summary.isna().sum().sum() == 0, "Unexpected NaNs found!"
print("✅ Case 2 complete and verified!")
display(tier_risk_summary)

In [ ]:
# ✍️ SHORTENED HIGH-SPEED VERSION FOR CASE 2:
import json
import pandas as pd

# BLOCK 1: Load and defensively clean in one chained expression
events = (
    pd.json_normalize(json.load(open('data/api_event_logs.json'))['data'])
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        risk_score=lambda d: pd.to_numeric(d['security_flags.risk_score'], errors='coerce').fillna(0),
        mfa_prompted=lambda d: d['security_flags.mfa_prompted'].astype(bool),
        mfa_passed=lambda d: d['security_flags.mfa_passed'].astype(bool)
    )
)
cust = (
    pd.read_csv('data/customers.csv')
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        account_tier=lambda d: d['account_tier'].astype(str).str.strip().str.title(),
        account_balance=lambda d: pd.to_numeric(d['account_balance'], errors='coerce').fillna(0)
    )
)

# 2. Filter attacks & pre-aggregate to customer grain in one step
attacked_cust=(events.query("`security_flags.risk_score` >= 0.85 or (`security_flags.mfa_prompted` and not `security_flags.mfa_passed`)")
    .groupby('customer_id',as_index=False)
    ['security_flags.risk_score']
    .mean()
)

# 3. Merge with customers & aggregate by tier
tier_risk_summary=(
    cust.merge(attacked_cust,on='customer_id',how='inner')
    .groupby('account_tier',as_index=False)
    .agg(
        attacked_customer_count=('customer_id','count'),
        total_balance_at_risk_usd=('account_balance','sum'),
        avg_attack_risk_score=('security_flags.risk_score','mean')
    )
    .round(2)
    .sort_values(by='total_balance_at_risk_usd',ascending=False)
    .reset_index(drop=True)
)

# 4. Quick 2-line sanity checks
assert tier_risk_summary['account_tier'].is_unique,"Duplicate tiers found!"
assert (tier_risk_summary['total_balance_at_risk_usd'] >= 0).all(),"Negative balance found!"

print("Case 2 complete and verified")
display(tier_risk_summary)

Case 2 complete and verified


,account_tier,attacked_customer_count,total_balance_at_risk_usd,avg_attack_risk_score
0,Standard,124,4361599.18,0.89
1,Silver,110,4264485.97,0.89
2,Platinum,112,4244921.90,0.89
3,VIP,120,4152763.28,0.89
4,Gold,97,3466080.97,0.89


---
## 🏢 Case 3 (Streaming JSONL & Device Fingerprinting): Rooted/Jailbroken Bot-Ring Identification

### 📌 Business Scenario
> **Stakeholder:** Principal Fraud Data Scientist  
> *"Our mobile security SDK outputs streaming telemetry into `device_telemetry.jsonl` (Newline-Delimited JSON). Ingest this streaming feed and identify compromised device sessions where `is_rooted_jailbroken == True` OR `network_type == 'Tor-Proxy'`. Join with `raw_transactions.csv` to calculate the total fraudulent transaction volume originating from compromised devices vs legitimate devices."*

In [5]:
# ✍️ YOUR CODE HERE FOR CASE 3:

---
## 🏢 Case 4 (Pipe-Separated PSV & XML Bureau Pulls): Synthetic Identity Ring Detection

### 📌 Business Scenario
> **Stakeholder:** Chief Credit Risk Officer (CCRO)  
> *"Synthetic identity rings create synthetic profiles using forged documents and manipulated credit files. Extract our regulatory identity verification logs `kyc_audit_records.psv` (handling `#` comment headers and pipe delimiters) and our credit bureau inquiries `credit_bureau_scores.xml`. Identify high-risk synthetic candidates defined as: `confidence_score < 0.70` OR `facial_match_pct < 60.0` OR `screening_flags` contains 'TAMPERED'/'SUSPICIOUS', while having `hard_inquiries_12m >= 4` or `delinquency_count_24m >= 2`. Output the top 15 highest-risk individuals ranked by a composite risk index."*

In [6]:
# ✍️ YOUR CODE HERE FOR CASE 4:

---
## 🏢 Case 5 (Fixed-Width ACH Clearing & NACHA Compliance): Unauthorized Return Rate Violations

### 📌 Business Scenario
> **Stakeholder:** Head of Payment Operations & Compliance  
> *"NACHA regulations mandate that merchants maintain an unauthorized return rate below 0.5% and overall NSF return rate below 15%. Ingest our NACHA fixed-width clearing file `ach_clearing_settlement.dat` using `pd.read_fwf()`. Calculate the total settled volume vs returned volume (`RETURNED_NSF`, `RETURNED_ACT_CLOSED`, `SUSPENDED_AML`). Join with `merchants.csv` to flag merchants exceeding NACHA risk thresholds and calculate their required mandatory rolling reserve deposit (5% of monthly volume)."*

In [7]:
# ✍️ YOUR CODE HERE FOR CASE 5:

---
## 🏢 Case 6 (YAML Sanctions Configuration & IP Intelligence): OFAC Sanctions Screening Docket

### 📌 Business Scenario
> **Stakeholder:** Global AML & FinCEN Compliance Officer  
> *"Our compliance team maintains active sanctions in `aml_sanctions_watchlist.yaml`. Load the YAML configuration to extract sanctioned countries and designated malicious IP prefixes. Scan all customer logins from `api_event_logs.json` and customer profiles from `customers.csv`. Generate an official FinCEN Escalation Docket containing all customers whose login IP matches a sanctioned IP prefix OR whose country of registration is under an active sanctions program."*

In [8]:
# ✍️ YOUR CODE HERE FOR CASE 6:

---
## 🏢 Case 7 (Multi-Table Cohort Lifecycle Analysis): Customer Cohort LTV vs. Dispute Loss Decay

### 📌 Business Scenario
> **Stakeholder:** Head of Customer Lifetime Value (CLV) & Growth Economics  
> *"Using `customers.csv`, `raw_transactions.csv`, and `disputes.csv`, group customers into quarterly onboarding cohorts (`account_created_at` converted to `YYYY-QX`). For each cohort, calculate total onboarded customer count, total transaction spending volume, total chargeback dispute loss, net interchange fee revenue (assuming 2.2% fee), and Net Cohort LTV per Customer."*

In [9]:
# ✍️ YOUR CODE HERE FOR CASE 7:

---
## 🏢 Case 8 (Dynamic Escrow Policy Engine): High-Velocity Merchant Risk & Auto-Hold Engine

### 📌 Business Scenario
> **Stakeholder:** Chief Risk Officer (CRO) & Merchant Operations Lead  
> *"Build a dynamic underwriting and settlement action engine for all active merchants in `merchants.csv`. Calculate each merchant's rolling 30-day transaction volume and dispute rate from `raw_transactions.csv` and `disputes.csv`. Implement vectorized risk rules:
> - **Tier 1 (Immediate Settlement Hold):** Dispute rate > 2.0% OR (Monthly Volume > $500k AND `is_chargeback_monitored == 1`).
> - **Tier 2 (7-Day Rolling Escrow):** Dispute rate between 1.0% and 2.0% OR Risk Rating == 'Extreme'.
> - **Tier 3 (Standard Auto-Payout):** All other compliant merchants.
> Generate an executive summary showing total merchants, total processing volume, and total frozen escrow funds across the 3 policy action tiers."*

In [10]:
# ✍️ YOUR CODE HERE FOR CASE 8: